# `agg` — Reference

`agg` aggregates numeric columns with grouping inherited from standalone `pt.by()` (or computes a whole-table summary when no grouping is set).

### Calling Styles

1. **Grouped aggregation**: `df.pt.by("species").pt.agg("mean")` or `df.pt.by("species").pt.agg(["sum", "mean", "n"])
2. **Keyword column mappings**: `df.pt.by("species").pt.agg(count="n", bill_length_mm="mean", body_mass_g=["sum", "mean"])
3. **Named output columns**: `df.pt.by("species").pt.agg(total="body_mass_g:sum")
4. **Whole-table summary**: `df.pt.agg("mean")` (computes grand totals across the entire table)

---


In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'src'))
if not os.path.exists(_src): _src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
tips = pt.sample_data['tips']
titanic = pt.sample_data['titanic']

## List `a` — `n` always goes first

In [2]:
# mean + n: regardless of list order, n always comes first before aggregated columns
(penguins
 .pt.select('species', 'island', 'bill_length_mm', 'body_mass_g')
 .pt.by('species', 'island')
 .pt.agg(a=['mean', 'n'])  # notice n comes first
)

,species,island,n,bill_length_mm,body_mass_g
0,Adelie,Biscoe,44,38.975000,3709.659091
1,Adelie,Dream,56,38.501786,3688.392857
2,Adelie,Torgersen,52,38.950980,3706.372549
3,Chinstrap,Dream,68,48.833824,3733.088235
4,Gentoo,Biscoe,124,47.504878,5076.016260


In [3]:
# Multiple numeric aggregations — column names get suffix: col_sum, col_mean, col_max
# also works with a single string if you only need one aggfunc
(penguins
 .pt.select('species', 'bill_length_mm', 'body_mass_g')
 .pt.by('species')
 .pt.agg(a=['sum', 'mean', 'max', 'n'])
)

,species,n,bill_length_mm_sum,bill_length_mm_mean,bill_length_mm_max,body_mass_g_sum,body_mass_g_mean,body_mass_g_max
0,Adelie,152,5857.5,38.791391,46.0,558800.0,3700.662252,4775.0
1,Chinstrap,68,3320.7,48.833824,58.0,253850.0,3733.088235,4800.0
2,Gentoo,124,5843.1,47.504878,59.6,624350.0,5076.016260,6300.0


## Keyword arguments — per-column control, output order follows parameter order


In [4]:
# Keyword arguments — mean for bill_length, sum+mean for body_mass, count named 'n'
(penguins
 .pt.by('species')
 .pt.agg(
     bill_length_mm='mean',
     body_mass_g=['sum', 'mean'],
     n='n',
 )
)

,species,bill_length_mm,body_mass_g_sum,body_mass_g_mean,n
0,Adelie,38.791391,558800.0,3700.662252,152
1,Chinstrap,48.833824,253850.0,3733.088235,68
2,Gentoo,47.504878,624350.0,5076.016260,124


In [5]:
# Place count first (n at front), then aggregations
# Output order follows parameter order
# Note: the count keyword can be named anything — 'cnt', 'count', even 'maddy'
(
    tips
    .pt.select('smoker', 'time', 'total_bill', 'tip')
    .pt.by('smoker', 'time')
    .pt.agg(cnt='n', total_bill='mean', tip=['mean', 'max'])
)


,smoker,time,cnt,total_bill,tip_mean,tip_max
0,Yes,Lunch,23,17.399130,2.834348,5.0
1,Yes,Dinner,70,21.859429,3.066000,10.0
2,No,Lunch,45,17.050889,2.673778,6.7
3,No,Dinner,106,20.095660,3.126887,9.0


In [6]:
# You can name the count column anything you like as a keyword argument
# Here maddy='n' creates a column called 'maddy' that holds the group count
(
    penguins
    .pt.select('species', 'island', 'body_mass_g')
    .pt.by('species')
    .pt.agg(maddy='n', body_mass_g='sum')
)


,species,maddy,body_mass_g
0,Adelie,152,558800.0
1,Chinstrap,68,253850.0
2,Gentoo,124,624350.0


In [7]:
# Multiple count columns as keyword arguments — each count gets its own name
(
    penguins
    .pt.select('species', 'island', 'body_mass_g')
    .pt.by('species')
    .pt.agg(count='n', countv2='n', body_mass_g='mean')
)


,species,count,countv2,body_mass_g
0,Adelie,152,152,3700.662252
1,Chinstrap,68,68,3733.088235
2,Gentoo,124,124,5076.016260


In [8]:
# median is a standard aggfunc — works in list form or as keyword argument
(
    penguins
    .pt.select('species', 'island', 'bill_length_mm', 'body_mass_g')
    .pt.by('species')
    .pt.agg(a=['median', 'mean', 'n'])
)


,species,n,bill_length_mm_median,bill_length_mm_mean,body_mass_g_median,body_mass_g_mean
0,Adelie,152,38.80,38.791391,3700.0,3700.662252
1,Chinstrap,68,49.55,48.833824,3700.0,3733.088235
2,Gentoo,124,47.30,47.504878,5000.0,5076.016260


## Handling Missing Group Keys (`dropna`)

In [9]:
# Default dropna=False keeps missing grouping keys in their own group (no silent data loss)
(penguins
 .pt.select('species', 'sex', 'bill_length_mm', 'body_mass_g')
 .pt.by('species', 'sex')
 .pt.agg(a=['mean', 'n'])
)

,species,sex,n,bill_length_mm,body_mass_g
0,Adelie,Female,73,37.257534,3368.835616
1,Adelie,Male,73,40.390411,4043.493151
2,Adelie,NaN,6,37.840000,3540.000000
3,Chinstrap,Female,34,46.573529,3527.205882
4,Chinstrap,Male,34,51.094118,3938.970588
5,Gentoo,Female,58,45.563793,4679.741379
6,Gentoo,Male,61,49.473770,5484.836066
7,Gentoo,NaN,5,45.625000,4587.500000


In [10]:
# Pass dropna=True to exclude groups with NaN keys
(penguins
 .pt.select('species', 'sex', 'bill_length_mm', 'body_mass_g')
 .pt.by('species', 'sex')
 .pt.agg(a=['mean', 'n'], dropna=True)
)

,species,sex,n,bill_length_mm,body_mass_g
0,Adelie,Female,73,37.257534,3368.835616
1,Adelie,Male,73,40.390411,4043.493151
2,Chinstrap,Female,34,46.573529,3527.205882
3,Chinstrap,Male,34,51.094118,3938.970588
4,Gentoo,Female,58,45.563793,4679.741379
5,Gentoo,Male,61,49.473770,5484.836066


## `n`-only aggregation — count with no numeric columns needed

In [11]:
# Count only — useful for frequency tables on categorical columns
(penguins
 .pt.select(exclude_dtype='number')
 .pt.by('species', 'island')
 .pt.agg(a='n')
)

,species,island,n
0,Adelie,Biscoe,44
1,Adelie,Dream,56
2,Adelie,Torgersen,52
3,Chinstrap,Dream,68
4,Gentoo,Biscoe,124


## Pipeline Chaining: Filter -> Select -> Aggregation -> Sort

Because  operations return DataFrames, you can chain them seamlessly with standard Pandas operations:

In [12]:
(
    penguins
    .pt.filter('body_mass_g > 3000')
    .pt.select('species', 'sex', 'body_mass_g')
    .pt.by('species', 'sex')
    .pt.agg(a=['mean', 'max', 'n'])
    .sort_values('body_mass_g_mean', ascending=False)
)

,species,sex,n,body_mass_g_mean,body_mass_g_max
6,Gentoo,Male,61,5484.836066,6300.0
5,Gentoo,Female,58,4679.741379,5200.0
7,Gentoo,NaN,4,4587.500000,4875.0
1,Adelie,Male,73,4043.493151,4775.0
4,Chinstrap,Male,34,3938.970588,4800.0
2,Adelie,NaN,4,3681.250000,4250.0
3,Chinstrap,Female,32,3572.656250,4150.0
0,Adelie,Female,65,3424.615385,3900.0


## Columns with Spaces & String Specifications

When column names have spaces, passing them as keyword arguments (`[total bill]='mean'`) is forbidden by Python syntax.

`agg` solves this by accepting **string mapping specifications** (matching the CLI `-agg` flag), where column names with spaces are enclosed in square brackets `[col]`. No dictionary unpacking `**{...}` is required:

In [13]:
# Create DataFrame with spaced column names
tips_spaced = tips.rename(columns={'total_bill': 'total bill'})

# Single string mapping with bracketed spaced columns
(
    tips_spaced
    .pt.by('smoker')
    .pt.agg("tip = mean, [total bill] = mean, n = n")
)

,smoker,tip,total bill,n
0,Yes,3.008710,20.756344,93
1,No,2.991854,19.188278,151


In [14]:
# Multiple positional string specifications
(
    tips_spaced
    .pt.by('smoker')
    .pt.agg("tip = mean", "[total bill] = mean", "n = n")
)

,smoker,tip,total bill,n
0,Yes,3.008710,20.756344,93
1,No,2.991854,19.188278,151


In [15]:
# Custom output names and bracketed source column
(
    tips_spaced
    .pt.by('smoker')
    .pt.agg("[avg bill] = [total bill]:mean, [total tip] = tip:sum, [table count] = n")
)

,smoker,avg bill,total tip,table count
0,Yes,20.756344,279.81,93
1,No,19.188278,451.77,151


In [16]:
# Complete pipeline: filter -> select -> agg with bracketed spaced columns across every verb
(
    tips
    .rename(columns={'total_bill': 'total bill'})
    .pt.filter("[total bill] > 10", day=['Sat', 'Sun'], time='Dinner', size='>= 2')
    .pt.select('smoker', 'tip', '[total bill]')
    .pt.by('smoker')
    .pt.agg("tip = mean, [total bill] = mean, n = n")
)

,smoker,tip,total bill,n
0,Yes,3.087719,23.232281,57
1,No,3.225464,20.705876,97
